In [1]:
import os
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
src_path = os.environ.get('PROJECT_SRC')
os.chdir(src_path)

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import heat_pattern
import FES_scaling_factor
import geopandas as gpd
import FES_shapefile_LAs_matching
import LA_overlaps_within_gird_squares
import heat_tech_distribution
from tech_half_hourly_profile import scaling_half_hourly_profiles_grid_sqs, base_heat_pattern_per_technology
from Grid_squares_heat_tech_demand import calculate_heat_demand_for_grid_squares

In [3]:
weather_year = 2010
future_year = 2050
FES_year = '2022'
FES_scenario = 'LW'
grid_id = '-3.5_56.0'

In [4]:


LA_shapefile = gpd.read_file(r'data\LA_UK\new_version\UK_LAs_Boundaries_files\LAs_shapefile_2021\LAD_DEC_2021_GB_BFC.shp')
grid_squares =  gpd.read_file(r'data\LA_UK\new_version\data_call\UK_Atlite_Grid_sq.gpkg')

In [5]:
air_temp_df = pd.read_csv(f'data/LA_UK/new_version/air_temp_new/air_temp_{weather_year}.csv', index_col=0)
grid_sqs = [str(gid).strip() for gid in air_temp_df.columns]

In [6]:
LA_overlaps = LA_overlaps_within_gird_squares.calc_LA_overlap_within_grid_square(grid_squares,
                                                                                LA_shapefile,
                                                                                weather_year)
LA_overlaps  

OSError: Cannot save file into a non-existent directory: 'C:\Users\s2583817\OneDrive - University of Edinburgh\Documents\Projects\PhD_Project\Python\PyPSA-GB_Domestic_Heat\data\LA_UK\new_version'

In [ ]:
df_merged = heat_tech_distribution.merge_shapefile_FES_heating_tech_file(grid_squares, 
                                                                             LA_overlaps, 
                                                                             future_year,
                                                                             FES_year, 
                                                                             FES_scenario,
                                                                             weather_year)

df_merged

In [ ]:
FES_scaling = FES_scaling_factor.FES_scaling_factor_calc(future_year, FES_year, FES_scenario) 
base_pattern = base_heat_pattern_per_technology()
FES_scaling

In [ ]:
total_dwellings = calculate_heat_demand_for_grid_squares(LA_overlaps, weather_year, future_year, FES_year, FES_scenario, LA_shapefile, grid_squares)


In [ ]:
total_dwellings_total = total_dwellings.sum()
total_dwellings_total

In [ ]:
pattern_profile_path = f"data/LA_UK/new_version/outputs/heat_demand_profiles_patterns/HD_heat_patterns_{weather_year}"
scaled_profiles_dict = scaling_half_hourly_profiles_grid_sqs(
pattern_profile_path,
total_dwellings,
weather_year,
future_year,
FES_year,
FES_scenario,
output_dir=None)

In [ ]:
pattern_profile_path = f"data/LA_UK/new_version/outputs/heat_demand_profiles_patterns/HD_heat_patterns_{weather_year}"
file_name = f"({grid_id})_{weather_year}.csv"


In [ ]:
  # Example grid ID, replace with actual grid ID as needed
file_name = f"half_hourly_heat_profile_{grid_id}_{weather_year}_{FES_year}_{FES_scenario}_{future_year}.csv"

folder = f"data/LA_UK/new_version/outputs/heating_profiles/{weather_year}_{FES_year}_{FES_scenario}_{future_year}"
file_path = os.path.join(folder, file_name)

hd_df = pd.read_csv(file_path, index_col=0)
hd_df.index = pd.to_datetime(hd_df.index)

In [ ]:
hd_df.columns

In [ ]:
hd_df

In [ ]:
plt.style.use('ggplot')
hd_df.plot.area(figsize=(20,8), cmap='tab20')

In [ ]:
plt.style.use('ggplot')
hd_df['2050-01-01 00:00:00': '2050-01-03 00:00:00'].plot.area(figsize=(20,8), stacked=True, cmap='tab20')

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
# 🚨 Load your GeoDataFrames properly
# Assuming these are already loaded:
# grid_squares, LA_shapefile, LA_overlaps

# ✅ 1. Ensure geometry is correctly set and CRS is assigned

def prepare_gdf(gdf, name, crs="EPSG:27700"):
    if 'geometry' not in gdf.columns:
        raise ValueError(f"{name} has no geometry column.")

    gdf.set_geometry('geometry', inplace=True)

    if gdf.crs is None:
        print(f"⚠️ CRS not set for {name}. Setting to {crs}.")
        gdf.set_crs(crs, inplace=True)
    elif gdf.crs.to_string() != crs:
        print(f"🔄 Reprojecting {name} to {crs}.")
        gdf.to_crs(crs, inplace=True)

    return gdf
# Ensure CRS and geometry are set
grid_squares = prepare_gdf(grid_squares, "grid_squares")
LA_shapefile = prepare_gdf(LA_shapefile, "LA_shapefile")

# Create a label ID for each LA
LA_shapefile = LA_shapefile.reset_index(drop=True)
LA_shapefile["LA_ID"] = LA_shapefile.index + 1  # Start from 1

# Plot setup
fig, ax = plt.subplots(figsize=(12, 14))

# Plot Local Authorities
LA_shapefile.plot(
    ax=ax,
    color="whitesmoke",
    edgecolor="black",
    linewidth=0.8,
)

# Plot grid squares (lighter)
grid_squares.boundary.plot(
    ax=ax,
    edgecolor="red",
    linestyle="--",
    linewidth=0.3,
    label="Grid Squares"
)

# Annotate LA numbers on map (centroids)
for idx, row in LA_shapefile.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        text=str(row['LA_ID']),
        xy=(centroid.x, centroid.y),
        fontsize=5,
        color='darkblue',
        ha='center',
        va='center'
    )

# Build a lookup legend for IDs ↔ LA names
legend_table = LA_shapefile[['LA_ID', 'LAD21NM']].sort_values('LA_ID')

# Final touches
plt.title("UK Local Authorities with Numeric Labels", fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

# Display the legend table separately
import pandas as pd
import IPython.display as display
display.display(legend_table.head(30))  # show top 30 LAs only


In [ ]:
# 🟠 Step 1 — Create heat_df from your scaled profiles
heat_totals = {
    grid_id: profile.sum().sum()
    for grid_id, profile in scaled_profiles_dict.items()
}
heat_df = pd.DataFrame(list(heat_totals.items()), columns=["Grid_ID", "total_demand"])
heat_df['Grid_ID'] = heat_df['Grid_ID'].astype(str).str.strip()

# 🟠 Step 2 — Match grid_squares Grid_IDs to the same format
grid_squares['Grid_ID'] = grid_squares.apply(
    lambda row: f"{float(row['x'])}_{float(row['y'])}", axis=1
).astype(str).str.strip()
# from shapely.geometry import MultiPolygon

# # Combine all Local Authority shapes into one 'landmass'
# land_union = LA_shapefile.unary_union

# # Flag whether each grid square intersects land
# grid_squares['is_land'] = grid_squares.geometry.intersects(land_union)

# # Keep only land-intersecting grid squares
# grid_squares = grid_squares[grid_squares['is_land']]

# 🟠 Step 3 — Merge heat demand into grid square geometries
merged = grid_squares.merge(heat_df, on="Grid_ID", how="left")

# Optional: check how many matched
print(f"✅ {merged['total_demand'].notna().sum()} of {len(merged)} grid squares have heating data.")
# Make sure CRS match
# grid_squares = grid_squares.to_crs("EPSG:27700")
# LA_shapefile = LA_shapefile.to_crs("EPSG:27700")

# # Spatial intersection: get the actual overlapping areas
# LA_overlaps = gpd.overlay(LA_shapefile, grid_squares, how='intersection')

fig, ax = plt.subplots(figsize=(10, 12))

# 🔥 Heatmap — draw *first* but with transparency
merged.dropna(subset=["total_demand"]).plot(
    ax=ax,
    column="total_demand",
    cmap="YlOrRd",
    legend=True,
    edgecolor="none",  # no outlines here
    alpha=0.9  # semi-transparent
)

# 🟥 Grid square boundaries — lighter and thinner
grid_squares.boundary.plot(
    ax=ax,
    edgecolor="grey",
    linestyle="--",
    linewidth=0.2,
    label="Grid Squares"
)

# 🗺️ Local Authorities — draw *last* so they are clear and sharp
LA_shapefile.boundary.plot(
    ax=ax,
    edgecolor="black",
    linewidth=0.8,
    label="Local Authority Boundaries"
)

# 🔵 Overlap regions — if any, plotted gently
# if not LA_overlaps.empty and "geometry" in LA_overlaps.columns:
#     LA_overlaps.plot(
#         ax=ax,
#         color="blue",
#         alpha=0.2,
#         label="Overlap Regions"
#     )

# Final styling
plt.title("Heating Demand Heatmap per Grid Square", fontsize=14)
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()



In [ ]:
from HD_Plots import (
    plot_week_of_heat_profiles,
    plot_total_daily_demand,
    plot_daily_avg_profile_one_grid,
    plot_heatmap_one_grid,
    plot_selected_grids_daily_demand,
    plot_boxplot_half_hourly_distribution,
    plot_tech_heatmaps
)


In [ ]:
plot_week_of_heat_profiles(
    folder_path="outputs/heating_profiles/2022_LW_2050",
    start_date="2050-01-01",
    days=5,
    num_grids=2
)


In [ ]:
for grid_id, obj in scaled_profiles_dict.items():
    print(f"{grid_id}: {type(obj)}")
    break  # just print the first one


In [ ]:
plot_total_daily_demand(scaled_profiles_dict)  # All grid squares

plot_total_daily_demand(scaled_profiles_dict, grid_id="-3.5_56.0")  # Just one grid


In [ ]:
plot_heatmap_one_grid(scaled_profiles_dict, grid_id="-3.5_56.0")
# for the City of Edinburgh Gas Boiler demand, Oil Boiler demand, and Electric resistive are zero

In [ ]:
plot_selected_grids_daily_demand(scaled_profiles_dict, grid_ids=["-3.5_56.0", "-3.25_55.75", "-3.0_56.0"])
#  3 grid_sqs of 5 grid_sqs for The City of Edinburgh

In [ ]:
plot_boxplot_half_hourly_distribution(scaled_profiles_dict)  # all grids

plot_boxplot_half_hourly_distribution(scaled_profiles_dict, grid_ids=["-3.5_56.0"])  # one grid

plot_boxplot_half_hourly_distribution(scaled_profiles_dict, grid_ids=["-3.5_56.0", "-3.25_55.75"])  # several
# "-3.5_55.75", "-3.25_56.0", "-3.0_56.0"(City of Edinburgh(5 gs)), 

In [ ]:
plot_tech_heatmaps(scaled_profiles_dict)